# Bases de datos en Python

## Índice
1. [Acceso a bases de datos relacionales](#sql)
2. [Crear bases de datos locales](#crear)

<a id="sql"></a>
## Acceso a bases de datos relacionales

Podemos acceder a bases de datos SQL con la librería `pymysql`. Para instalarla escribimos en Anaconda Prompt:  
`conda install -c anaconda pymysql`

In [1]:
!pip install pymysql

In [5]:
!pip install mysql-connector-python
!pip install mysqlclient
!pip install mysql

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ----- ---------------------------------- 2.1/16.4 MB 10.7 MB/s eta 0:00:02
   ---------- ----------------------------- 4.5/16.4 MB 11.2 MB/s eta 0:00:02
   ------------ --------------------------- 5.2/16.4 MB 11.0 MB/s eta 0:00:02
   ---------------- ----------------------- 6.6/16.4 MB 7.9 MB/s eta 0:00:02
   --------------------- ------------------ 8.9/16.4 MB 8.7 MB/s eta 0:00:01
   ---------------------------- ----------- 11.5/16.4 MB 9.2 MB/s eta 0:00:01
   --------------------------------- ------ 13.9/16.4 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------  16.3/16.4 MB 9.7 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 9.6 MB/s eta 0:00:00


In [14]:
import pymysql

#### Ejemplo 1
Vamos a conectarnos a la base de datos NBA, que contiene estadísticas de partidos de una temporada.  
Utilizaremos los siguientes parámetros:  
* servidor: relational.fit.cvut.cz
* usuario: guest
* contraseña: relational  
* base de datos: NBA  

Vamos a crear una conexión con la base de datos

In [15]:
database_host = 'relational.fit.cvut.cz'
username = 'guest'
password = 'relational'
database_name = 'NBA'

db = pymysql.connect(host=database_host,
                    user=username,
                    password=password,
                    database=database_name,
                    port=3306)

OperationalError: (2003, "Can't connect to MySQL server on 'relational.fit.cvut.cz' (timed out)")

El método `read_sql()` de pandas nos permite crear dataframes a partir de queries.

In [12]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

query = 'SELECT * FROM Actions'
df = pd.read_sql(query, db)
df

NameError: name 'db' is not defined

Obtenemos los jugadores con más de 5 asistencias en algún partido

In [5]:
query = '''
SELECT PlayerName, Player.PlayerId, Assists
FROM Actions
JOIN Player
ON Actions.PlayerId = Player.PlayerId
WHERE Assists>5
'''

Asistencias5 = pd.read_sql(query, db)
Asistencias5

,PlayerName,PlayerId,Assists
0,Nicolas Batum,1,7
1,LaMarcus Aldridge,2,6
2,Wesley Matthews,4,6
3,Damian Lillard,5,8
4,Kent Bazemore,14,6
...,...,...,...
64,Kyrie Irving,323,8
65,Jameer Nelson,335,7
66,Tony Parker,351,8
67,Goran Dragic,363,8


Obtenemos el TOP 10 jugadores con más asistencias en la temporada

In [6]:
query = '''
SELECT PlayerName, SUM(Assists) AS TotalAssists
FROM Actions
JOIN Player
ON Actions.PlayerId = Player.PlayerId
GROUP BY PlayerName
ORDER BY TotalAssists DESC
LIMIT 10
'''

Asistencias = pd.read_sql(query, db)
Asistencias

,PlayerName,TotalAssists
0,Chris Paul,27.0
1,Kemba Walker,25.0
2,Brandon Jennings,22.0
3,Ty Lawson,21.0
4,Demar DeRozan,20.0
5,Ramon Sessions,17.0
6,Andre Miller,17.0
7,DJ Augustin,16.0
8,Lebron James,16.0
9,John Wall,16.0


Obtenemos el TOP 10 Equipos con más puntuación media por partido

In [7]:
query = '''
SELECT TeamName, AVG(TotalPointsGame) AS AvgPoints
FROM(
    SELECT TeamName, GameId, SUM(Points) AS TotalPointsGame
    FROM Actions
    JOIN Team
    ON Actions.TeamId = Team.TeamId
    GROUP BY TeamName, GameId
    )TABLA1
GROUP BY TeamName
ORDER BY AvgPoints DESC
LIMIT 10 
'''

Puntos = pd.read_sql(query, db)
Puntos

,TeamName,AvgPoints
0,Portland Trail Blazers,124.0000
1,Cleveland Cavaliers,119.0000
2,Dallas Mavericks,116.5000
3,Golden State Warriors,112.5000
4,Los Angeles Clippers,111.0000
5,Charlotte Bobcats,108.3333
6,Phoenix Suns,108.0000
7,Los Angeles Lakers,107.0000
8,Denver Nuggets,107.0000
9,Washington Wizards,106.0000


#### Ejemplo 2
Vamos a conectarnos a la base de datos de los empleados de una empresa  
* servidor: relational.fit.cvut.cz
* usuario: guest
* contraseña: relational  
* base de datos: employees

In [ ]:
database_host = 'relational.fel.cvut.cz'
username = 'guest'
password = 'ctu-relational'
database_name = 'employee'

db = pymysql.connect(host=database_host,
                    user=username,
                    password=password,
                    database=database_name,
                    port=3306)

OperationalError: (2003, "Can't connect to MySQL server on 'relational.fel.cvut.cz' (timed out)")

In [5]:
pd.read_sql('SHOW TABLES', db)

,Tables_in_employee
0,departments
1,dept_emp
2,dept_manager
3,employees
4,salaries
5,titles


Obtener el salario máximo, mínimo y medio por género y cargo

In [10]:
pd.read_sql('SELECT * FROM employees LIMIT 10', db)

,emp_no,birth_date,first_name,last_name,gender,hire_date
0,10001,1953-09-02,Georgi,Facello,M,1986-06-26
1,10002,1964-06-02,Bezalel,Simmel,F,1985-11-21
2,10003,1959-12-03,Parto,Bamford,M,1986-08-28
3,10004,1954-05-01,Chirstian,Koblick,M,1986-12-01
4,10005,1955-01-21,Kyoichi,Maliniak,M,1989-09-12
5,10006,1953-04-20,Anneke,Preusig,F,1989-06-02
6,10007,1957-05-23,Tzvetan,Zielinski,F,1989-02-10
7,10008,1958-02-19,Saniya,Kalloufi,M,1994-09-15
8,10009,1952-04-19,Sumant,Peac,F,1985-02-18
9,10010,1963-06-01,Duangkaew,Piveteau,F,1989-08-24


In [11]:
pd.read_sql('SELECT * FROM titles LIMIT 10', db)

,emp_no,title,from_date,to_date
0,10001,Senior Engineer,1986-06-26,9999-01-01
1,10002,Staff,1996-08-03,9999-01-01
2,10003,Senior Engineer,1995-12-03,9999-01-01
3,10004,Engineer,1986-12-01,1995-12-01
4,10004,Senior Engineer,1995-12-01,9999-01-01
5,10005,Senior Staff,1996-09-12,9999-01-01
6,10005,Staff,1989-09-12,1996-09-12
7,10006,Senior Engineer,1990-08-05,9999-01-01
8,10007,Senior Staff,1996-02-11,9999-01-01
9,10007,Staff,1989-02-10,1996-02-11


In [12]:
pd.read_sql('SELECT * FROM salaries LIMIT 10', db)

,emp_no,salary,from_date,to_date
0,10001,60117,1986-06-26,1987-06-26
1,10001,62102,1987-06-26,1988-06-25
2,10001,66074,1988-06-25,1989-06-25
3,10001,66596,1989-06-25,1990-06-25
4,10001,66961,1990-06-25,1991-06-25
5,10001,71046,1991-06-25,1992-06-24
6,10001,74333,1992-06-24,1993-06-24
7,10001,75286,1993-06-24,1994-06-24
8,10001,75994,1994-06-24,1995-06-24
9,10001,76884,1995-06-24,1996-06-23


In [13]:
query = '''
SELECT title, gender, MIN(salary), AVG(salary), MAX(salary), COUNT(emp.emp_no)
FROM employees emp
JOIN titles tit
ON emp.emp_no = tit.emp_no
JOIN salaries sal
ON emp.emp_no = sal.emp_no
WHERE sal.to_date = '9999-01-01'
AND tit.to_date = '9999-01-01'
GROUP BY title, gender
'''

Salarios = pd.read_sql(query, db)
Salarios

,title,gender,MIN(salary),AVG(salary),MAX(salary),COUNT(emp.emp_no)
0,Senior Engineer,M,39285,70869.9085,140784,51533
1,Staff,F,38936,67282.4599,137875,10090
2,Senior Staff,M,39012,80735.4795,158220,49232
3,Senior Engineer,F,39476,70753.8341,138273,34406
4,Senior Staff,F,39227,80662.9816,152710,32792
5,Engineer,F,39519,59617.3549,115444,12412
6,Engineer,M,38942,59592.9683,130939,18571
7,Staff,M,39186,67362.1754,133577,15436
8,Assistant Engineer,F,39469,57495.9861,106340,1440
9,Technique Leader,F,39812,67369.0734,144434,4866


In [14]:
query = '''
SELECT COUNT(DISTINCT emp_no)
FROM titles
WHERE title = 'Manager'
AND to_date = '9999-01-01'
'''

pd.read_sql(query,db)

,COUNT(DISTINCT emp_no)
0,9


<a id="crear"></a>
## Crear bases de datos relacionales

SQLite es un sistema de gestión de bases de datos basado en SQL optimizado para entornos pequeños como aplicaciones móviles. Puede integrarse con Python gracias a la librería `sqlite3`, incluida por defecto en las versiones más recientes de Python.

In [16]:
import sqlite3

In [17]:
# Conexión a una base de datos
conn = sqlite3.connect('my_database.sqlite')
cursor = conn.cursor()

In [18]:
# Crear tablas
cursor.execute('''
CREATE TABLE SCHOOL
(ID INT PRIMARY KEY NOT NULL,
 NAME TEXT NOT NULL,
 AGE INT NOT NULL,
 CITY CHAR(50),
 MARKS INT
)
''');

In [19]:
# Insertar valores
cursor.execute('''
INSERT INTO SCHOOL (ID, NAME, AGE, CITY, MARKS)
VALUES (1, 'Luis', 24, 'Madrid', 8)
''')

cursor.execute('''
INSERT INTO SCHOOL (ID, NAME, AGE, CITY, MARKS)
VALUES (2, 'Ana', 34, 'Bilbao', 8)
''')



In [20]:
conn.commit()

In [21]:
# Ejecutar queries
pd.read_sql('SELECT * FROM SCHOOL', conn)

,ID,NAME,AGE,CITY,MARKS
0,1,Luis,24,Madrid,8
1,2,Ana,34,Bilbao,8


In [22]:
# Crear tablas a partir de dataframes
Salarios.to_sql('SALARIOS', conn, index=False)

NameError: name 'Salarios' is not defined

In [ ]:
# Listar tablas
pd.read_sql('SELECT name FROM sqlite_master WHERE type="table"', conn)

,name
0,SCHOOL
1,SALARIOS


In [ ]:
# Borrar tablas
cursor.execute('DROP TABLE IF EXISTS SALARIOS')

In [ ]:
pd.read_sql('SELECT name FROM sqlite_master WHERE type="table"', conn)

,name
0,SCHOOL


In [ ]:
# Actualizar registros
cursor.execute('UPDATE SCHOOL SET MARKS=4 WHERE ID=2')
conn.commit()

In [ ]:
pd.read_sql('SELECT * FROM SCHOOL', conn)

,ID,NAME,AGE,CITY,MARKS
0,1,Luis,24,Madrid,8
1,2,Ana,34,Bilbao,4


In [ ]:
# Borrar registros
cursor.execute('DELETE FROM SCHOOL WHERE ID=2')
conn.commit()

In [ ]:
pd.read_sql('SELECT * FROM SCHOOL', conn)

,ID,NAME,AGE,CITY,MARKS
0,1,Luis,24,Madrid,8


In [ ]:
conn.close()